# Fundamental Ratio Features

## Overview

This notebook demonstrates the public functions in `fundamental_ratio_features` using compact synthetic financial ratios.
- Problem: financial statements and prices need to be converted into comparable value, profitability, leverage, efficiency, and shareholder-return features.
- Approach: provide FinanceToolkit-shaped synthetic ratio tables and combine the selected representative ratios into ticker-period rows.
- Fundamental Ratio Features: It produces P/B, ROE, operating margin, debt-to-equity, asset turnover, cash-flow yield, dividend yield, and market capitalization.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src.data_preprocessing.fundamental_ratio_features import (
    collect_fundamental_ratio_features,
)

## Collect Fundamental Ratio Features

This cell defines a synthetic FinanceToolkit ratio provider and collects feature rows.
- Each quarterly series uses plausible positive values for a profitable large-cap company.
- Market capitalization rises gradually while leverage remains bounded and cash-flow yield stays positive.
- The synthetic provider follows the public `toolkit.ratios` interface used by the feature module.

In [ ]:
periods = pd.Index(["2024Q1", "2024Q2", "2024Q3", "2024Q4"], name="period")

class SyntheticRatios:
    def __init__(self):
        self.values = {
            "get_price_to_book_ratio": [6.2, 6.4, 6.1, 6.3],
            "get_return_on_equity": [1.42, 1.45, 1.47, 1.50],
            "get_operating_margin": [0.30, 0.31, 0.31, 0.32],
            "get_debt_to_equity_ratio": [1.65, 1.60, 1.58, 1.55],
            "get_asset_turnover_ratio": [1.02, 1.04, 1.06, 1.08],
            "get_free_cash_flow_yield": [0.031, 0.033, 0.034, 0.036],
            "get_dividend_yield": [0.005, 0.005, 0.005, 0.005],
            "get_market_cap": [2.7e12, 2.8e12, 2.9e12, 3.0e12],
        }

    def __getattr__(self, name):
        def get_ratio(**kwargs):
            return pd.DataFrame(
                [self.values[name]],
                index=pd.Index(["AAPL"], name="ticker"),
                columns=periods,
            )
        return get_ratio

class SyntheticToolkit:
    ratios = SyntheticRatios()

fundamental_features = collect_fundamental_ratio_features(SyntheticToolkit())

assert fundamental_features["free_cash_flow_yield"].gt(0).all()
assert fundamental_features["market_cap"].is_monotonic_increasing
display(fundamental_features)

This cell visualizes profitability, leverage, and scale in the synthetic feature table.
- ROE and operating margin increase slightly, representing improving profitability.
- Debt-to-equity declines, representing a modestly less leveraged balance sheet.
- Market capitalization is plotted in trillions of dollars so its scale is interpretable beside ratio features.

In [ ]:
plot_data = fundamental_features.set_index("period")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_data[["return_on_equity", "operating_margin", "debt_to_equity"]].plot(
    ax=axes[0],
    marker="o",
)
axes[0].set_title("Profitability And Leverage")
axes[0].set_ylabel("Ratio")
(plot_data["market_cap"] / 1e12).plot(ax=axes[1], marker="o", color="tab:blue")
axes[1].set_title("Market Capitalization")
axes[1].set_ylabel("USD trillions")
plt.tight_layout()
plt.show()